**Overview**

This notebook shows how I built a simple conversational assistant that can remember past messages within a session. Instead of replying to each message independently, the assistant keeps track of the conversation so responses feel more connected and meaningful.




**Motivation**

Most basic chatbots only look at the current message and ignore what was said before. That often makes the conversation feel disconnected or repetitive.
To improve this, I added a session-based memory system that stores previous messages and uses them while generating responses. This helps the assistant understand context and respond more naturally.



**Core Components**

Language Model – handles response generation
Prompt Structure – controls how inputs and history are passed to the model
Session Memory Manager – keeps track of conversation history for each session
Message Storage – stores messages separately for different users or sessions

**Setup Environment**

In [1]:
!pip install langchain langchain-community langchain-openai python-dotenv

**Loading OpenAI API key**

In [4]:
import os
os.environ["OPENAI_API_KEY"] = "your_openai_api_key"

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

**Initialize Language Model**

In [6]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=800
)

**Create Session-Based Memory Store**

In [7]:
session_memory = {}

def load_history(session_key: str):
    if session_key not in session_memory:
        session_memory[session_key] = ChatMessageHistory()
    return session_memory[session_key]

**Define Prompt Template**

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant that remembers previous messages and gives clear, short responses."),
    MessagesPlaceholder(variable_name="chat_log"),
    ("human", "{user_input}")
])

**Build Processing Chain**

In [9]:
pipeline = prompt | llm

**Attach Memory to Chain**

In [10]:
chat_agent = RunnableWithMessageHistory(
    pipeline,
    load_history,
    input_messages_key="user_input",
    history_messages_key="chat_log"
)

**Example Usage**

In [12]:
session_id = "session_001"

response1 = chat_agent.invoke(
    {"user_input": "Hi, I moved to Texas recently"},
    config={"configurable": {"session_id": session_id}}
)
print("Siri:", response1.content)

response2 = chat_agent.invoke(
    {"user_input": "Where did I say I moved?"},
    config={"configurable": {"session_id": session_id}}
)
print("Siri:", response2.content)

Siri: Hi again! How's your experience in Texas so far?
Siri: You mentioned that you moved to Texas recently.


In [13]:
session_id = "session_001"

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    response = chat_agent.invoke(
        {"user_input": user_input},
        config={"configurable": {"session_id": session_id}}
    )

    print("Siri:", response.content)

You: Hello, How are you?
Siri: Hello! I'm just a program, but I'm here and ready to help you. How can I assist you today?
You: I want to book tickets from seattle to dallas today
Siri: You can book tickets online through airline websites or travel booking platforms like Expedia, Kayak, or Google Flights. Do you need help with anything specific?
You: Can you book for me?
Siri: I can't make bookings directly, but I can guide you through the process if you'd like!
You: Okay can you let me know what are the flights available ?
Siri: I can't check real-time flight availability, but you can visit airline websites or travel booking sites like Expedia, Kayak, or Google Flights to see available flights from Seattle to Dallas. Would you like tips on how to search?
You: You: Suggest the cheapest way to travel between those cities
Siri: The cheapest way to travel between Seattle and Dallas is usually by bus or train, but it takes longer. For flights, booking in advance and using budget airlines ca

In [15]:
hist = session_memory[session_id].messages
if len(hist) > 10:
    session_memory[session_id].messages = hist[-10:]

In [17]:
print("\nConversation History:")

for message in session_memory[session_id].messages:
    print(f"{message.type}: {message.content}")


Conversation History:
human: Can you book for me?
ai: I can't make bookings directly, but I can guide you through the process if you'd like!
human: Okay can you let me know what are the flights available ?
ai: I can't check real-time flight availability, but you can visit airline websites or travel booking sites like Expedia, Kayak, or Google Flights to see available flights from Seattle to Dallas. Would you like tips on how to search?
human: You: Suggest the cheapest way to travel between those cities
ai: The cheapest way to travel between Seattle and Dallas is usually by bus or train, but it takes longer. For flights, booking in advance and using budget airlines can help you find lower fares. You can also check for deals on travel comparison websites. Would you like more details on any specific option?
human: Give me a simple plan for booking tickets
ai: Here’s a simple plan for booking your tickets:

1. **Choose Your Dates**: Decide when you want to travel.
2. **Search for Flights*